In [ ]:
%pip install -q ultralytics kagglehub
import re
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
from ultralytics import YOLO
import kagglehub

In [ ]:
# Configuration
SEED = 42
EPOCHS = 30
IMG = 640
ALIASES = {"hogpulm": "hogplum", "lichi": "litchi"}

In [ ]:
# Download dataset
DATA_DIR = Path(kagglehub.dataset_download(
    "afsananadia/fruits-images-dataset-object-detection"
))
OUT = Path("fruits_yolo")

In [ ]:
# Prepare folders
for split in ["train", "val"]:
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

In [ ]:
# Clean class names
def clean_name(stem):
    name = re.sub(r"^\d+_", "", stem).lower().replace(" ", "")
    return ALIASES.get(name, name)

In [ ]:
# Collect classes
all_classes = set()
for img in DATA_DIR.rglob("*.jpg"):
    all_classes.add(clean_name(img.stem))
classes = sorted(all_classes)
print("Fruit classes:", classes)

In [ ]:
# Prepare YOLO dataset
counts = {"train": 0, "val": 0}
folders = {"train": "Train File", "val": "Test File"}
for split, folder in folders.items():
    for img in (DATA_DIR / folder).rglob("*.jpg"):
        txt = img.with_suffix(".txt")
        if not txt.exists():
            continue
        cls = clean_name(img.stem)
        if cls not in classes:
            continue
        cls_id = classes.index(cls)
        shutil.copy2(img, OUT / "images" / split / img.name)
        lines = []
        for line in txt.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                lines.append(f"{cls_id} " + " ".join(parts[1:5]))
        (OUT / "labels" / split / f"{img.stem}.txt").write_text(
            "\n".join(lines)
        )
        counts[split] += 1

In [ ]:
# Create YAML file
yaml = (
    f"path: {OUT.resolve().as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
)
yaml += "\n".join(f"  {i}: {c}" for i, c in enumerate(classes))
(OUT / "data.yaml").write_text(yaml)
print("\nDataset prepared!")
print("Training images:", counts["train"])
print("Validation images:", counts["val"])
if not classes or counts["train"] == 0 or counts["val"] == 0:
    raise ValueError("Dataset preparation failed. Check the dataset folders and labels.")

In [ ]:
# Load YOLOv8 model
model = YOLO("yolov8n.pt")

In [ ]:
# Train model
model.train(
    data=str((OUT / "data.yaml").resolve()),
    epochs=EPOCHS,
    imgsz=IMG,
    batch=-1,
    workers=2,
    seed=SEED,
    patience=10,
    project="runs",
    name="fruit_detection",
    exist_ok=True
)

In [ ]:
# Evaluate model
save_dir = Path(model.trainer.save_dir)
metrics = model.val(data=str((OUT / "data.yaml").resolve()))
print("\n========== MODEL RESULTS ==========")
print("Precision:", round(metrics.box.mp, 4))
print("Recall:", round(metrics.box.mr, 4))
print("mAP50:", round(metrics.box.map50, 4))
print("mAP50-95:", round(metrics.box.map, 4))

In [ ]:
# Display training graphs
for name in ["results.png", "confusion_matrix.png"]:
    path = save_dir / name
    if path.exists():
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(path))
        plt.title(name)
        plt.axis("off")
        plt.show()

In [ ]:
# Display validation predictions
val_images = sorted((OUT / "images" / "val").glob("*.jpg"))
samples = val_images[:8]
results = model.predict(
    [str(p) for p in samples],
    imgsz=IMG,
    conf=0.25,
    verbose=False
)
plt.figure(figsize=(12, 7))
for i, result in enumerate(results):
    plt.subplot(2, 4, i + 1)
    plt.imshow(result.plot()[..., ::-1])
    plt.axis("off")
plt.suptitle("Fruit Object Detection Predictions")
plt.tight_layout()
plt.show()

In [ ]:
# Save best model
Path("outputs").mkdir(exist_ok=True)
shutil.copy2(
    save_dir / "weights" / "best.pt",
    "outputs/fruits_yolov8n_det.pt"
)
print("\nBest model saved: outputs/fruits_yolov8n_det.pt")
print("Fruit Object Detection Completed!")

In [ ]:
# Detect fruits in a new image
def detect_image(path, conf=0.25):
    result = model.predict(
        str(path), imgsz=IMG, conf=conf, verbose=False
    )[0]
    plt.figure(figsize=(6, 6))
    plt.imshow(result.plot()[..., ::-1])
    plt.axis("off")
    plt.show()
    return [
        (
            result.names[int(box.cls)],
            round(float(box.conf), 3),
            tuple(box.xyxy[0].tolist())
        )
        for box in result.boxes
    ]